In [9]:
# Visualização dos dados de entrada
import pandas as pd
arq = './dados/integras_experimento_summa_novos.parquet'
arq = './dados/teste_integras_experimento_summa.parquet'
df = pd.read_parquet(arq)
#df[df['fold'] == 12]
#df[(df['integra'] == '') & (df['fold'] == 12)]
#df

dfs = df[['seq_documento_acordao','ano','fold']]
dfs.to_csv('./saida/filtro_teste_completo.csv')
#print(df)

# fold 12
dfs = df[(df['integra'] != '') & (df['fold'] == 12)]
dfs = dfs[['seq_documento_acordao','ano','fold']]
dfs.to_csv('./saida/filtro_teste_fold_12.csv')

# fold 11
dfs = df[(df['integra'] != '') & (df['fold'] == 11)]
dfs = dfs[['seq_documento_acordao','ano','fold']]
dfs.to_csv('./saida/filtro_teste_fold_11.csv')

# fold 1 a 10
dfs = df[(df['integra'] != '') & (df['fold'].between(1, 10) )]
dfs = dfs[['seq_documento_acordao','ano','fold']]
dfs.to_csv('./saida/filtro_teste_fold_1a10.csv')


In [5]:
# União dos dataframes e contagem de tokens

import pandas as pd
import json
#arq = './saida/saida_1.5b.parquet'
origem = './dados/integras_experimento_summa_novos.parquet'
resposta = './saida/saida_oa_gpt5.parquet'
df_origem = pd.read_parquet(origem)
df_resposta = pd.read_parquet(resposta)

df_resposta['chave'] = df_resposta['chave'].astype(int)
df_merge = df_origem.merge(df_resposta, left_on='seq_documento_acordao', right_on='chave', how='inner')

#dados = dados[-10:].copy().reset_index()
print(df_merge.iloc[0])
q = len(df_merge)

print('-='*40)

# fold 12
dfs = df_merge[df_merge['fold'] == 12]
entrada = 0
saida = 0
for i, row in dfs.iterrows():
    dados = row['resumo']
    dados = json.loads(dados)
    #print(row)
    entrada += dados.get('prompt_tokens',0)
    saida += dados.get('completion_tokens',0)

print('Registros:', len(dfs), 'de', len(df_merge))
print(f'Tokens de entrada:', entrada)    # R$5 
print(f'Tokens de saída:', saida)        # R$20
print(f'Total Sabiá:', (entrada*5 + saida*20) / 1000000)

sg_classe                                                            AREsp
pasta                                                 ./saidas/<<tipo>>_01
fold                                                                     1
dt_publicacao                                                   2024-12-17
sg_ramo_direito                                                         TR
num_registro                                                  202301681297
ano                                                                   2024
seq_documento_acordao                                            286006309
num_ministro                                                        1187.0
integra                  ACÓRDÃO\nVistos e relatados estes autos em que...
index                                                                  NaN
chave                                                            286006309
resumo                   {"prompt_tokens": 9824, "completion_tokens": 3...
resposta                 

In [7]:
import pandas as pd
import os
arquivos = ['saida_oa_gpt5.parquet', 'saida_qwen7b.parquet', 'saida_or_235b.parquet', 'saida_sabia4.parquet']

for arq in arquivos:
    arq = os.path.join('./saida', arq)
    df = pd.read_parquet(arq)
    df = df[df['integra'] != '']
    print(f'Arquivo: {arq} com {len(df)} registros')

 

KeyError: 'integra'

In [11]:
import json
resposta = dados['resposta'][q-1]
doc = dados['chave'][q-1]
print('Documento:', doc)
try:
    j = json.loads(resposta)
    print(j)
except Exception as e:
    print('ERRO:', e)
    print(resposta)

Documento: 286006126
ERRO: Expecting value: line 1 column 1 (char 0)
```json
{
  "Tipo": "ResAcordao",
  "Materia": "Alegada violação ao art. 1.022 do CPC/2015",
  "Dispositivo": "Conheço do recurso, porquanto presentes os seus pressupostos intrínsecos e extrínsecos de admissibilidade.",
  "DataJulgamento": "2024-04-15",
  "Partes": [
    {
      "Tipo": "Apelante",
      "Nome": "Estado de Mato Grosso"
    }
  ],
  "Temas": [
    {
      "Ponto": "Alegada violação ao art. 1.022 do CPC/2015",
      "Preliminar": "Não",
      "Argumentos": [
        "A Corte de origem dirimiu, fundamentadamente, a matéria submetida à sua apreciação, manifestando-se acerca dos temas necessários ao integral deslinde da controvérsia, não havendo omissão, contradição, obscuridade ou erro material, afastando-se, por conseguinte, a alegada violação ao art. 1.022 do CPC/2015."
      ],
      "Normas": [
        "Artigos 489, § 1º, IV, e 1.022, II e III, e parágrafo único, II, do CPC/2015"
      ],
      "Juris

In [ ]:
# Mostra o último registro com erro
# import json
erros = dados[dados['erro'] > '']
if len(erros) > 0:
    resposta = erros['resposta'].iloc[-1]
    registro = erros['num_registro'].iloc[-1]
    doc = erros['seq_documento_acordao'].iloc[-1]
    try:
        j = json.loads(resposta)
        print(j)
    except Exception as e:
        print('ERRO:', e)
        print(resposta)
else:
    print('nenhum erro encontrado')